In [ ]:
import os
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

## Configuration and Imports

In this section, we import the necessary libraries and define the configuration parameters for our experiment, including paths and hyperparameters.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import models, transforms
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# Add functions directory to path
sys.path.append(os.path.abspath("functions"))

from functions.dataset import COVIDCXNetDataset
from functions.train import train
from functions.evaluation import eval_on_metrics

# Constants
DATA_DIR = "/home/furkan/Projects/Datasets/COVID-CXNet/"
CSV_PATH = "/home/furkan/Projects/Datasets/COVID-CXNet/covidx_merged.csv"
SEEDS = [42, 123, 2024]
BATCH_SIZE = 32
EPOCHS = 5
LR = 0.0001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {DEVICE}")

## Data Loader Preparation

We prepare the data transformations and create the data loaders for the training, validation, and test splits using the `COVIDCXNetDataset`.

In [ ]:
# Transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Datasets
train_dataset = COVIDCXNetDataset(csv_file=CSV_PATH, root_dir=DATA_DIR, transform=transform, split="train")
val_dataset = COVIDCXNetDataset(csv_file=CSV_PATH, root_dir=DATA_DIR, transform=transform, split="val")
test_dataset = COVIDCXNetDataset(csv_file=CSV_PATH, root_dir=DATA_DIR, transform=transform, split="test")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

os.makedirs("logs", exist_ok=True)
os.makedirs("models", exist_ok=True)

## Seed Comparison Experiment

We iterate through different random seeds, train a ResNet-18 model for each, and collect the final test metrics to analyze the impact of randomness.

In [ ]:
results = []

for seed in SEEDS:
    print(f"\n{'='*20}\nRunning Experiment with Seed: {seed}\n{'='*20}")
    set_seed(seed)
    
    # Initialize Model
    model = models.resnet18(weights='DEFAULT')
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.to(DEVICE)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)
    
    # Train
    save_path = f'models/best_model_seed_{seed}.pth'
    log_path = f'logs/training_seed_{seed}.log'
    
    metrics = train(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=DEVICE,
        save_path=save_path,
        num_epochs=EPOCHS,
        patience=3,
        log_path=log_path
    )
    
    # Evaluate on Test Set
    print(f'Evaluating Seed {seed} on Test Set...')
    model.load_state_dict(torch.load(save_path))
    
    model.eval()
    y_true = []
    y_pred = []
    with torch.no_grad():
        for imgs, lbls in test_loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            outputs = model(imgs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(lbls.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
    
    from sklearn.metrics import accuracy_score, f1_score
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    
    results.append({
        'seed': seed,
        'test_accuracy': acc,
        'test_f1': f1
    })
    
    print(f'Seed {seed} - Test Acc: {acc:.4f}, Test F1: {f1:.4f}')

## Results Analysis

Finally, we visualize the comparison of performance metrics across different seeds.

In [ ]:
results_df = pd.DataFrame(results)
print('\nSummary Table:')
print(results_df)

# Plotting
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.barplot(x='seed', y='test_accuracy', data=results_df)
plt.title('Test Accuracy across Seeds')
plt.ylim(results_df['test_accuracy'].min() - 0.05, results_df['test_accuracy'].max() + 0.05)

plt.subplot(1, 2, 2)
sns.barplot(x='seed', y='test_f1', data=results_df)
plt.title('Test F1 Score across Seeds')
plt.ylim(results_df['test_f1'].min() - 0.05, results_df['test_f1'].max() + 0.05)

plt.tight_layout()
plt.show()